In [1]:
import os
os.chdir('../')
os.getcwd()

'/home/minh_khai/skin_disease/skin-disease'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class RagConfig:
    pdf_path:           Path
    chroma_dir:         Path
    embedding_model:    str
    chunk_size:         int
    chunk_overlap:      int
    top_k:              int

In [3]:
import os
from core.constants import *
from core import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_rag_config(self) -> RagConfig:
        config = self.config.rag_config
        params = self.params.rag_params

        return RagConfig(
            pdf_path        = Path(config.pdf_path),
            chroma_dir      = Path(config.chroma_dir),
            embedding_model = params.embedding_model,
            chunk_size      = params.chunk_size,
            chunk_overlap   = params.chunk_overlap,
            top_k           = params.top_k,
        )

In [4]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from core.logger import logger

class RagIngestor:
    def __init__(self, config: RagConfig):
        self.config = config
        self.embedding = HuggingFaceEmbeddings(model_name=config.embedding_model)
        
    def ingest(self):
        logger.info(f"Loading PDF: {self.config.pdf_path}")
        loader = PyPDFLoader(str(self.config.pdf_path))
        documents = loader.load()
        logger.info(f"Loaded {len(documents)} pages")
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap,
        )
        chunks = splitter.split_documents(documents)
        logger.info(f"Split into {len(chunks)} chunks")
        
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embedding,
            persist_directory=str(self.config.chroma_dir),
        )
        logger.info(f"Persisted vector store to {self.config.chroma_dir}")
        return vectorstore

/tmp/ipykernel_6993/4256371170.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/minh_khai/skin_disease/skin-disease/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
try:
    cfg_manager = ConfigurationManager()
    rag_config  = cfg_manager.get_rag_config()
    ingestor    = RagIngestor(rag_config)
    vectorstore = ingestor.ingest()
except Exception as e:
    raise e

[2026-06-20 04:47:44,308: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-20 04:47:44,311: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-20 04:47:44,313: INFO: __init__: created directory at: artifacts]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2013.24it/s]


[2026-06-20 04:48:22,106: INFO: 4256371170: Loading PDF: data/DaLieu.pdf]


Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 207 0 (offset 0)
Ignoring wrong pointing object 734 0 (offset 0)
Ignoring wrong pointing object 883 0 (offset 0)


[2026-06-20 04:48:35,035: INFO: 4256371170: Loaded 330 pages]
[2026-06-20 04:48:35,058: INFO: 4256371170: Split into 721 chunks]
[2026-06-20 04:48:46,838: INFO: 4256371170: Persisted vector store to artifacts/agent/chroma_db]


In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=str(rag_config.chroma_dir), embedding_function=embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": rag_config.top_k})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3604.95it/s]
